In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
import os
os.listdir('/content/drive/MyDrive/Lehaiphong_AI/research')

['level1']

In [ ]:

%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/layer.py

import numpy as np
import torch
import torch.nn as nn

def _layer(x):
  layer=nn.Conv2d(in_channels=3,out_channels=64,kernel_size=3)
  return layer(x)


Writing /content/drive/MyDrive/Lehaiphong_AI/research/level1/layer.py


In [ ]:
!python /content/drive/MyDrive/Lehaiphong_AI/research/level1/code.py

25


In [ ]:
%cd /content/drive/MyDrive/Lehaiphong_AI/research/level1
from layer import _layer
import torch
x=torch.rand(3,32,32)
result=_layer(x)
print(result.shape)




/content/drive/MyDrive/Lehaiphong_AI/research/level1
torch.Size([64, 30, 30])


In [ ]:
%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/model_custom.py


Writing /content/drive/MyDrive/Lehaiphong_AI/research/level1/model_custom.py


In [8]:
%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/data_loader.py

import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.utils.data as data

#download data
def creat_dataloader():
  ROOT = './data'
  train_data = datasets.MNIST(
      root=ROOT,
      download = True,
      train= True
  )
  test_data = datasets.MNIST(
      root=ROOT,
      train = False,
      download = True
  )

  #split training

  valid_ratio = 0.9

  n_train_examples = int(len(train_data)*valid_ratio)
  n_valid_example = len(train_data)-n_train_examples

  train_data,valid_data = data.random_split(
      train_data,
      [n_train_examples,n_valid_example]
  )
  num_classes = len(train_data.dataset.classes)
  #compute mean anh std for normalization

  mean = train_data.dataset.data.float().mean()/255
  std = train_data.dataset.data.float().std()/255

  train_transforms = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(mean=[mean],std=[std])
  ])

  test_transforms = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(mean=[mean],std=[std])
  ])
  train_data.dataset.transform = train_transforms
  valid_data.dataset.transform = test_transforms

  #Create dataloader
  Batch_size = 256

  train_dataloader = data.DataLoader(
      train_data,
      shuffle = True,
      batch_size = Batch_size
  )
  valid_dataloader = data.DataLoader(
      valid_data,
      batch_size = Batch_size
  )
  return train_dataloader,valid_dataloader,num_classes

Overwriting /content/drive/MyDrive/Lehaiphong_AI/research/level1/data_loader.py


In [ ]:
%cd /content/drive/MyDrive/Lehaiphong_AI/research/level1
from data_loader import creat_dataloader

train_loader,valid_loader = creat_dataloader()

for image, labels in train_loader :
  img0 = image[1]
  lable0 = labels[1]
  print(img0.shape,lable0.shape)
  break

/content/drive/MyDrive/Lehaiphong_AI/research/level1
torch.Size([1, 28, 28]) torch.Size([])


In [ ]:
%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/model.py

import torch.nn as nn
class LenetClassifier(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.conv1 = nn.Conv2d(1,6,kernel_size = 5,padding='same')
    self.avgpol1 = nn.AvgPool2d(2,2)
    self.conv2 = nn.Conv2d(6,16,5)
    self.avgpol2 = nn.AvgPool2d(2,2)
    self.flaten = nn.Flatten()
    self.relu = nn.ReLU()
    self.linear0 = nn.Linear(16*5*5,120)
    self.linear1 = nn.Linear(120,84)
    self.linear2 = nn.Linear(84,num_classes)
  def forward(self,x):
    x = self.conv1(x)

    x = self.avgpol1(x)
    x = self.relu(x)
    x = self.conv2(x)
    x = self.avgpol2(x)
    x = self.relu(x)
    x = self.flaten(x)
    x = self.linear0(x)
    x = self.linear1(x)
    x = self.linear2(x)
    return x


Overwriting /content/drive/MyDrive/Lehaiphong_AI/research/level1/model.py


In [2]:
%cd /content/drive/MyDrive/Lehaiphong_AI/research/level1
from model import LenetClassifier
import torch
lenet_model = LenetClassifier(10)
x=torch.rand(256,1,28,28)
lenet_model(x).shape

/content/drive/MyDrive/Lehaiphong_AI/research/level1


torch.Size([256, 10])

In [5]:
import torch
model = LenetClassifier(10)
x = torch.rand(1,1,28,28)
result = model(x)
print(result.shape)

torch.Size([1, 10])


In [47]:
%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/train3_evalid.py

import time
import torch

def train_model(model,optimizer,criterion,train_dataloader,device,epoch=0,log_interval=50):
  model.train()
  total_acc,total_count = 0,0
  losses = []
  start_time = time.time()
  for idx, (inputs,labels) in enumerate(train_dataloader):
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    predictions = model(inputs)

    loss = criterion(predictions,labels)
    losses.append(loss.item())

    #backward
    loss.backward()
    torch.nn.utils.clip_grad_norm_( model.parameters () , 0.1)
    optimizer.step()
    total_acc += (predictions.argmax(1) == labels).sum().item()
    total_count += labels.size(0)

    if idx % log_interval ==0 and idx >0:
      elapsed = time.time() -start_time
      print(
          "|epoch {:3d}|{:5d}/{:5d} batches"
          " accurracy{:8.3f}" .format(epoch,idx,len(train_dataloader),total_acc/total_count)
       )
      total_acc , total_count = 0, 0
      start_time = time.time()
  epoch_acc = total_acc/total_count
  epoch_loss = sum(losses)/len(losses)
  return epoch_acc, epoch_loss,model
def evaluate(model,criterion, valid_dataloader,device):
  model.eval()
  total_acc, total_count = 0,0
  losses = []
  with torch.no_grad():
    for idx, (inputs,labels) in enumerate(valid_dataloader):
      inputs = inputs.to(device)
      labels = labels.to(device)

      predictions = model(inputs)

      loss = criterion(predictions , labels)

      losses.append(loss)

      total_acc += (predictions.argmax(1) == labels).sum().item()
      total_count += labels.size(0)

  epoch_acc = total_acc / total_count
  epoch_loss = sum(losses)/len(losses)
  return epoch_acc, epoch_loss




Overwriting /content/drive/MyDrive/Lehaiphong_AI/research/level1/train3_evalid.py


In [48]:
%%writefile /content/drive/MyDrive/Lehaiphong_AI/research/level1/run_train_lenet.py

from data_loader import creat_dataloader
from model import LenetClassifier
from train3_evalid import train_model,evaluate
import torch.nn as nn
import torch . optim as optim
import torch
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'
)
train_dataloader,valid_dataloader,num_classes = creat_dataloader()
lenet_model = LenetClassifier(num_classes)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters())
num_epoch = 10
save_model = '/content/drive/MyDrive/Lehaiphong_AI/research/level1'

def run_train_model_lenetmodel(num_classes,num_epoch,device,model,train_dataloader,valid_dataloader, criterion,optimizer):

  train_accs,train_losses = [],[]
  eval_accs,eval_losses = [],[]
  best_loss_eval =100
  best_loss_eval = 100
  model=model
  for epoch in range(1,num_epoch+1):
    epoch_start_time = time.time()

    train_acc,train_loss,model_final = train_model(model,optimizer,criterion,train_dataloader,device,epoch)
    train_accs.append(train_acc)
    train_losses.append(train_loss)

    eval_acc,eval_loss = evaluate(model_final,criterion,valid_dataloader,device)
    eval_accs.append(eval_acc)
    eval_losses.append(eval_loss)
    if eval_loss < best_loss_eval:
      torch.save(model.state_dict(),save_model + '/lenet_model.pt'
                 )
    print("epoch:",epoch,"eval_acc:",eval_acc,"eval_loss:",eval_loss)
  return eval_accs,eval_losses,train_accs,train_losses

run_train_model_lenetmodel(num_classes,num_epoch,device,lenet_model,train_dataloader,valid_dataloader, criterion,optimizer)



Overwriting /content/drive/MyDrive/Lehaiphong_AI/research/level1/run_train_lenet.py


In [ ]:
!python /content/drive/MyDrive/Lehaiphong_AI/research/level1/run_train_lenet.py

|epoch   1|   50/  211 batches accurracy   0.707
|epoch   1|  100/  211 batches accurracy   0.889
|epoch   1|  150/  211 batches accurracy   0.924
|epoch   1|  200/  211 batches accurracy   0.942
epoch: 1 eval_acc: 0.9405 eval_loss: tensor(0.1855)
|epoch   2|   50/  211 batches accurracy   0.957
|epoch   2|  100/  211 batches accurracy   0.962
|epoch   2|  150/  211 batches accurracy   0.964
|epoch   2|  200/  211 batches accurracy   0.968
epoch: 2 eval_acc: 0.965 eval_loss: tensor(0.1172)
|epoch   3|   50/  211 batches accurracy   0.967
|epoch   3|  100/  211 batches accurracy   0.973
